In [2]:
!python --version

Python 3.11.13


재구성

In [3]:
!pip uninstall -y tf-keras keras-nightly keras==3.* tensorflow==2.16.* tensorflow==2.17.* tensorflow==2.18.* tf-nightly


Found existing installation: keras 2.15.0
Uninstalling keras-2.15.0:
  Successfully uninstalled keras-2.15.0
Found existing installation: tensorflow 2.15.0.post1
Uninstalling tensorflow-2.15.0.post1:
  Successfully uninstalled tensorflow-2.15.0.post1


In [4]:
!pip -q install "numpy==1.26.4" "ml-dtypes==0.2.0" "h5py==3.10.0"


In [5]:
%env TF_USE_LEGACY_KERAS=1
!pip -q install "tensorflow==2.15.0.post1" "keras==2.15.0"


env: TF_USE_LEGACY_KERAS=1


In [6]:
!pip -q install --no-deps "deepctr==0.9.3"


재시작

In [1]:

import os, sys, types
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
import keras

# 내부 경로에 Keras 2 심볼 매핑
sys.modules["tensorflow.python.keras.layers"] = keras.layers
sys.modules["tensorflow.python.keras.initializers"] = keras.initializers

# init_ops_v2 대체 (DeepCTR가 참조)
from keras.initializers import TruncatedNormal, Constant, glorot_uniform
init_v2 = types.ModuleType("tensorflow.python.ops.init_ops_v2")
init_v2.TruncatedNormal = TruncatedNormal
init_v2.Constant = Constant
init_v2.glorot_uniform = glorot_uniform
sys.modules["tensorflow.python.ops.init_ops_v2"] = init_v2

print("TF:", tf.__version__, "| Keras:", keras.__version__, "→ shim ready")

TF: 2.15.0 | Keras: 2.15.0 → shim ready


In [2]:
import tensorflow as tf, keras, deepctr
print("TF:", tf.__version__)
print("Keras:", keras.__version__)
print("DeepCTR:", deepctr.__version__)

from deepctr.feature_column import SparseFeat, DenseFeat, get_feature_names
from deepctr.models import DeepFM

print("DeepCTR import OK")


TF: 2.15.0
Keras: 2.15.0
DeepCTR: 0.9.3
DeepCTR import OK


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import roc_auc_score, log_loss, average_precision_score
from deepctr.feature_column import SparseFeat, DenseFeat, VarLenSparseFeat, get_feature_names
from deepctr.models import DeepFM
import numpy as np


In [4]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

train= pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/CRT/train_part1.parquet' , engine= 'pyarrow')
test = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/CRT/test.parquet' , engine= 'pyarrow')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


test 열에 없는 열 train에서 버리기

In [5]:
target = 'clicked'

cols_to_keep = [c for c in train.columns if c in test.columns or c == target]

train = train[cols_to_keep]

train , test 데이터 타입 맞추기

- test 데이터 id열 제외하고 float32로




In [6]:
cols_to_convert = [c for c in test.columns if c not in ['seq' ,'ID']]
test[cols_to_convert]  = test[cols_to_convert].astype("float32")

메타데이터 저장

In [10]:
train["is_train"] = 1
test["is_train"]  = 0

test_id = test["ID"].copy()

In [11]:
all_data = pd.concat([train, test], ignore_index=True)

In [12]:
df = all_data.copy()

In [13]:
df.info(verbose = True , show_counts= True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4731183 entries, 0 to 4731182
Data columns (total 121 columns):
 #    Column        Non-Null Count    Dtype  
---   ------        --------------    -----  
 0    gender        4730247 non-null  float32
 1    age_group     4730247 non-null  float32
 2    inventory_id  4731183 non-null  float32
 3    day_of_week   4731183 non-null  float32
 4    hour          4731183 non-null  float32
 5    seq           4731183 non-null  object 
 6    l_feat_1      4731183 non-null  float32
 7    l_feat_2      4730247 non-null  float32
 8    l_feat_3      4731183 non-null  float32
 9    l_feat_4      4731183 non-null  float32
 10   l_feat_5      4731183 non-null  float32
 11   l_feat_6      4731183 non-null  float32
 12   l_feat_7      4731183 non-null  float32
 13   l_feat_8      4730247 non-null  float32
 14   l_feat_9      4731183 non-null  float32
 15   l_feat_10     4731183 non-null  float32
 16   l_feat_11     4731183 non-null  float32
 17   l_feat

inventory_id = 92 , 21 이상치로 간주후 삭제하려 했으나 아래 같은 방식으로 하면 test 데이터도 함께 날아가는 이슈 발생

따라서 어차피 unk(0)로 학습해야 할 바엔 표본이 충분한 92는 그대로 학습 , 21은 unk(0)으로 일단 ㄱㄱ

In [14]:
df = df.drop(['l_feat_20' , 'l_feat_23','l_feat_2','l_feat_24'], axis = 1 )
df = df.drop(['history_a_1' , 'history_a_2' , 'history_a_3'], axis= 1)
# df = df[~df['inventory_id'].isin([92, 21])]

In [15]:
is_train_num = pd.to_numeric(df["is_train"], errors="coerce")
tr_mask = is_train_num.eq(1)

# 전역 CTR + 베타-스무딩
p_global = df.loc[tr_mask, "clicked"].mean()
prior_s  = 300
alpha, beta = p_global*prior_s, (1-p_global)*prior_s

grp = df.loc[tr_mask, ["inventory_id","clicked"]].dropna(subset=["inventory_id"]).groupby("inventory_id")
cnt = grp.size()
pos = grp["clicked"].sum()
post_ctr = (pos.add(alpha, fill_value=0)) / (cnt.add(prior_s, fill_value=0))

# 보존 기준(튜닝 가능)
min_n = 500      # 표본 수 기준
delta = 0.010    # 전역 CTR 대비 1.0%p 이상 차이면 보존
keep_ids = post_ctr[(cnt >= min_n) & ((post_ctr - p_global).abs() >= delta)].index

# train-only map: 보존 ID만 개별 임베딩, 나머지는 UNK=0
mp_inv = {v:i+1 for i, v in enumerate(sorted(keep_ids))}
inv_vocab_size = (max(mp_inv.values()) + 1) if mp_inv else 1  # UNK만 있어도 최소 1

# 보조 플래그(선택) — 모델에 Dense로 같이 넣으면 도움될 수 있음
df["inv_is_rare"]  = (~df["inventory_id"].isin(keep_ids)).astype("int8")
df["inv_is_92_21"] = df["inventory_id"].isin([92,21]).astype("int8")


    max_len : 800
    topk : 1000
    min_count = 500

In [16]:
import numpy as np
import pandas as pd

MAX_LEN = 150
PAD_ID = 0  # 0은 PAD, 실제 토큰은 +1 오프셋

def build_seq_padded_len(series: pd.Series, max_len: int, *, offset: int = 1, ignore_neg: bool = True):
    """
    series: 콤마 구분 문자열("9,18,269,...") 컬럼
    offset=1: 모델 전처리처럼 +1 오프셋(0은 PAD로 예약)
    return: seq_padded(int32, [N,max_len]), seq_len(int32, [N]), vocab_size(int)
    """
    # 미리 결과 배열을 한 번에 할당(메모리/속도 핵심)
    N = len(series)
    seq_padded = np.zeros((N, max_len), dtype=np.int32)
    seq_len    = np.zeros(N, dtype=np.int32)
    max_id     = 0

    # 판다스 오버헤드 줄이기: 바로 넘파이 배열로
    # astype(str)을 쓰면 NaN -> 'nan' 문자열이 되므로, 아래에서 비어 있으면 건너뜀
    vals = series.to_numpy(copy=False)

    for i in range(N):
        s = vals[i]
        if s is None or (isinstance(s, float) and np.isnan(s)):
            # 빈 시퀀스
            continue

        # 문자열로 캐스팅 (np.fromstring은 공백을 무시하므로 replace 불필요)
        text = s if isinstance(s, str) else str(s)
        if not text:
            continue

        # C 가속 파싱: 매우 빠름. 실패하면 size=0
        arr = np.fromstring(text, sep=',', dtype=np.int64)
        if arr.size == 0:
            continue

        if ignore_neg:
            # 음수 제거(있다면)
            arr = arr[arr >= 0]
            if arr.size == 0:
                continue

        if offset:
            # +1 오프셋 (0=PAD 유지)
            arr = arr + offset

        # 트렁케이팅: 최신 항목을 남기고 앞을 자름 (pre-truncating)
        L = arr.size
        if L > max_len:
            arr = arr[-max_len:]
            L = max_len

        # 패딩된 행에 앞쪽부터 복사 (post-padding)
        # arr는 int64이므로 복사 시 자동 캐스팅 → 비용 적음
        seq_padded[i, :L] = arr
        seq_len[i] = L

        # vocab_size 계산용 최대 id 갱신 (한 번에 끝)
        amax = int(arr.max()) if L > 0 else 0
        if amax > max_id:
            max_id = amax

    vocab_size = int(max_id + 1)  # PAD 포함
    return seq_padded, seq_len, vocab_size

# 사용 예시
seq_padded, seq_len, vocab_size = build_seq_padded_len(df["seq"], MAX_LEN, offset=1, ignore_neg=True)

# # df에 길이만 저장
# df["seq_len"] = seq_len


In [17]:
PAD_ID   = 0
MAX_LEN  = 150
OFFSET   = 1


tr_mask = df["is_train"].eq(1)
te_mask = df["is_train"].eq(0)


# 1) train만으로 vocab “fit”
seq_tr_pad, seq_tr_len, vocab_seq = build_seq_padded_len(
    df.loc[tr_mask, "seq"], MAX_LEN, offset=OFFSET, ignore_neg=True
)

# 2) test는 같은 규칙으로 transform만 + OOV 클램핑
seq_te_pad, seq_te_len, _ = build_seq_padded_len(
    df.loc[te_mask, "seq"], MAX_LEN, offset=OFFSET, ignore_neg=True
)

# train에서 결정한 vocab_seq를 기준으로, 범위 밖 토큰(>= vocab_seq)은 0으로
seq_tr_pad = np.where(seq_tr_pad < vocab_seq, seq_tr_pad, PAD_ID).astype("int32")
seq_te_pad = np.where(seq_te_pad < vocab_seq, seq_te_pad, PAD_ID).astype("int32")

# 길이 저장(원하면 df에도 반영)
df.loc[tr_mask, "seq_len"] = seq_tr_len
df.loc[te_mask, "seq_len"] = seq_te_len


### 피처 열 생성


> DeepCTR에서 SparseFeat는 내부적으로 Embedding Lookup을 하기 때문에, 입력값은 반드시 0 ~ (vocabulary_size-1) 범위의 연속 정수 인덱스여야 한다.

>
    SparseFeat("inventory_id", vocabulary_size=16) 같이 정의하면, DeepCTR은 입력값을 0~15 정수 인덱스로 간주한다.

    그런데 실제 데이터는 2.0, 36.0, 37.0, …, 95.0처럼 흩어져 있음.

    이걸 그대로 Embedding lookup에 넣으면 → 인덱스 범위 초과 오류 또는 embedding index mismatch 발생.

> label encoding으로 맞춘다.


<br>


순서형의 경우 임베딩은 순서 정보를 기억하지 않는다.
>
    조회만 한다: e_k = Embedding[k] — k는 의미 있는 수가 아니라 “키”.

    순열 불변성: 첫 레이어가 ŷ = W·e_k + b일 때, 라벨을 임의로 섞고(순열 P) 임베딩과 가중치를 같이 섞으면 ŷ가 그대로.
    즉 모델은 라벨 순서에 무관하게 동치 해를 가짐.

    그래디언트 독립: 각 카테고리 벡터가 독립적으로 업데이트되어 연속성/단조성이 보장되지 않음. “2는 1과 3 사이”라는 규칙을 스스로 학습하리란 보장이 없다.

    그래서 임베딩으로 처리하면 ‘명목형처럼’ 취급되고, 순서(ordinal) 정보는 구조적으로 전달되지 않는다.

> 두 개의 표현을 동시에 사용 - SparseFeat(범주형) 과 DenseFeat(순서형 숫자) 둘 다 넣기.



<BR>

hour , day_of_week 열에 관하여

>

    day_of_week: 순서 중요, 월~일 주기성 있음

    hour: 시간대도 순서형이라 DenseFeat 가능. 다만 주기성(23시→0시)이 있어서 사인/코사인 변환.




< 계획 >

1. 결측치 보강


2. **hour, day_of_week**

 >
    사인/코사인 변환 + 이중표현(Sparse: 원래 카테고리도 함께 넣어 요일/시간대별 개별 패턴 포착)

    Sparse: 원래 카테고리도 함께 넣어 요일/시간대별 개별 패턴 포착.
>
    장점: 주기성(23→0의 연속성) + 카테고리별 임베딩 패턴을 동시에 잡는다.



3. 연속형 추정 열들

>

    l_feat_3, l_feat_27, feat_e_4, feat_a_1, feat_a_3, feat_a_4, feat_a_8, feat_a_13, feat_a_16, feat_a_18
>
    Dense: 결측치 imputer(median 등) → MinMaxScaler 후 그대로 입력.

    Sparse(해시): 결측치 보강 후 -> 바로  처리


> 이중화의 경우 과적합의 위험이 있으니 embedding_dim을 작게(4~8), dnn_dropout, l2_reg_embedding 등을 적절히 사용

In [18]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler

####### 1번
ord_cols  = ["hour", "day_of_week", "age_group"]

cont_cols = ["l_feat_3","l_feat_27","feat_e_4",
             "feat_a_1","feat_a_3","feat_a_4","feat_a_8",
             "feat_a_13","feat_a_16","feat_a_18"]

cat_cols =  ["gender", "inventory_id", "l_feat_14"]

seq_col = 'seq'
label_col = 'clicked'
seq_len_col = 'seq_len'

ban = set(ord_cols + cont_cols + cat_cols + [seq_col, label_col, seq_len_col, "is_train", "ID"])

# 진또베기 연속형들
extra_cont_cols = [c for c in df if c not in ban]

# 순서형 + 연속형
cont_all_cols = cont_cols + extra_cont_cols

# dense용 전체 (ord + cont + 찐연속형)
num_all_cols = ord_cols + cont_all_cols


# train / test 분리
tr_mask = df["is_train"].eq(1)
te_mask = df["is_train"].eq(0)


# train으로만 fit : test 결측치 대체할 통계 학습
imp_ord = SimpleImputer(strategy="most_frequent").fit(df.loc[tr_mask , ord_cols])
imp_cont = SimpleImputer(strategy="median").fit(df.loc[tr_mask, cont_all_cols])

Xord_tr  = imp_ord.transform(df.loc[tr_mask,ord_cols])
Xcont_tr = imp_cont.transform(df.loc[tr_mask,cont_all_cols])

Xnum_tr = np.hstack([Xord_tr , Xcont_tr])


# Dense용 스케일러 - train 단계에서 fit만해서 따로 저장
scaler = MinMaxScaler().fit(Xnum_tr)


# age , inventoy_id 임베딩
mp_age = {v:i+1 for i,v in enumerate(sorted(df.loc[tr_mask,"age_group"].dropna().unique()))}
age_vocab_size = max(mp_age.values()) + 1

mp_inv = {v:i+1 for i,v in enumerate(sorted(df.loc[tr_mask,"inventory_id"].dropna().unique()))}
inv_vocab_size = max(mp_inv.values()) + 1



# train에서만 mp 생성
nuniq    = {c: np.unique(Xcont_tr[:, cont_all_cols.index(c)]).size for c in cont_cols}
map_cols = [c for c, u in nuniq.items() if u <= 15]  # 임계값 10~15 권장

mp_cont = {
    c: {v: i+1 for i, v in enumerate(np.unique(Xcont_tr[:, cont_all_cols.index(c)]))}
    for c in map_cols
}

test 데이터에 결측치가 없어도 전처리 산출물에 정의를 해야함 -> 모델 입력 만들 때 keyEorror

> 결측치 유무는 상관 없이 정의하면 입력에도 존재해야 한다.


>

    age_group, inventory_id 등은 train에서만 mp = {value: i+1} 만들고,
    test는 map(...).fillna(0)로 OOV를 0에 보냄.

    vocabulary_size = max(mp.values()) + 1 로 맞춤

In [19]:
def make_dense_sparse(
    _df,
    mask,
    *,
    ord_fit_cols,           # fit에 사용한 순서형 리스트(예: ["hour","day_of_week","age_group"])
    cont_fit_cols,          # fit에 사용한 연속형 전체(cont_cols + extra_cont_cols)
    num_fit_cols,           # ord_fit_cols + cont_fit_cols
    imp_ord, imp_cont,      # SimpleImputer(most_frequent/median), train으로만 fit된 것
    scaler,                 # MinMaxScaler, train으로만 fit된 것
    map_cols, mp_cont       # 저카디널 연속형만 정수 매핑 정보(둘 다 train 기준)
):
    # --- 1) 같은 컬럼/순서로 transform (fit과 100% 동일) ---
    Xord  = imp_ord.transform(_df.loc[mask, ord_fit_cols])
    Xcont = imp_cont.transform(_df.loc[mask, cont_fit_cols])
    Xnum  = np.hstack([Xord, Xcont]).astype(np.float32)
    Xnum_s = scaler.transform(Xnum).astype(np.float32)

    dense = pd.DataFrame(Xnum_s, columns=num_fit_cols, index=_df.index[mask])



    h_idx  = ord_fit_cols.index("hour")
    dow_idx = ord_fit_cols.index("day_of_week")
    hour_vals = Xord[:, h_idx]
    dow_vals  = Xord[:, dow_idx]

    dense["hour_sin"] = np.sin(2*np.pi*hour_vals/24)
    dense["hour_cos"] = np.cos(2*np.pi*hour_vals/24)
    dense["dow_sin"]  = np.sin(2*np.pi*dow_vals/7)
    dense["dow_cos"]  = np.cos(2*np.pi*dow_vals/7)

    dense_cols = num_fit_cols + ["hour_sin","hour_cos","dow_sin","dow_cos"]




    sparse = pd.DataFrame(index=_df.index[mask])

    if "gender" in _df.columns:
        g = _df.loc[mask, "gender"].astype("float64")
        sparse["gender"] = g.where(g.isin([1, 2]), np.nan).fillna(0).astype("int32")


    # 0=UNK, 1..24 / 1..7 / 1..(K) 형태로 통일
    sparse["hour_cat"]        = (_df.loc[mask, "hour"].astype("float64").fillna(-1).astype("int32") + 1)
    sparse["day_of_week_cat"] = (_df.loc[mask, "day_of_week"].astype("float64").fillna(-1).astype("int32") + 1)


    sparse["age_group_cat"]    = _df.loc[mask,"age_group"].map(mp_age).fillna(0).astype("int32")
    sparse["inventory_id_cat"] = _df.loc[mask,"inventory_id"].map(mp_inv).fillna(0).astype("int32")



    # l_feat_14: 있으면 해시 문자열로 추가
    has_l14 = False
    if "l_feat_14" in _df.columns:
        sparse["l_feat_14"] = _df.loc[mask, "l_feat_14"].astype("string").fillna("UNKNOWN")
        has_l14 = True



    # 저카디널 연속형만 정수 매핑(임퓨트된 값 기준)
    for c in map_cols:
        j = cont_fit_cols.index(c)
        v = pd.Series(Xcont[:, j], index=_df.index[mask])
        sparse[c + "_cat"] = v.map(mp_cont[c]).fillna(0).astype("int32")




    # 최종 sparse_cols 조립
    base_sparse = ["hour_cat","day_of_week_cat","age_group_cat","inventory_id_cat"]
    if "gender" in sparse.columns:
        base_sparse = ["gender"] + base_sparse
    if has_l14:
        base_sparse.append("l_feat_14")

    sparse_cols = base_sparse + [c + "_cat" for c in map_cols]

    return dense[dense_cols], sparse[sparse_cols], dense_cols, sparse_cols



ORD_FIT_COLS  = ["hour","day_of_week","age_group"]   # inventory_id, l_feat_14는 제외(=sparse 전용)
# CONT_FIT_COLS는 숫자 dtype만: cont_cols + extra_cont_cols(숫자)
ban = set(ORD_FIT_COLS + cont_cols + ["gender","inventory_id","l_feat_14",
                                      seq_col, label_col, seq_len_col, "is_train", "ID"])
extra_cont_cols = [c for c in df.select_dtypes(include=["number"]).columns if c not in ban]
CONT_FIT_COLS = cont_cols + extra_cont_cols
NUM_FIT_COLS  = ORD_FIT_COLS + CONT_FIT_COLS

# (이미 위에서) imp_ord/imp_cont/scaler는 반드시 위 리스트로 train에서 fit 완료돼 있어야 함

dense_tr, sparse_tr, dense_cols, sparse_cols = make_dense_sparse(
    df, tr_mask,
    ord_fit_cols=ORD_FIT_COLS, cont_fit_cols=CONT_FIT_COLS, num_fit_cols=NUM_FIT_COLS,
    imp_ord=imp_ord, imp_cont=imp_cont, scaler=scaler,
    map_cols=map_cols, mp_cont=mp_cont
)
dense_te, sparse_te, _, _ = make_dense_sparse(
    df, te_mask,
    ord_fit_cols=ORD_FIT_COLS, cont_fit_cols=CONT_FIT_COLS, num_fit_cols=NUM_FIT_COLS,
    imp_ord=imp_ord, imp_cont=imp_cont, scaler=scaler,
    map_cols=map_cols, mp_cont=mp_cont
)



In [20]:
HASH_BUCKET = 200_000
MAX_LEN = 150

EMBED_DIM = 8 # 임시로 통일

def _vocab_from_int(s):
    # s: pd.Series(int). 반드시 0 이상.
    return int(s.max()) + 1 if len(s) else 1


sparse_fixed = [
    SparseFeat('gender',            vocabulary_size=_vocab_from_int(sparse_tr['gender']),            embedding_dim=EMBED_DIM, dtype='int32'),
    SparseFeat('inventory_id_cat',  vocabulary_size= inv_vocab_size,                                 embedding_dim= 16,       dtype='int32'),
    SparseFeat('hour_cat',          vocabulary_size=_vocab_from_int(sparse_tr['hour_cat']),          embedding_dim=EMBED_DIM, dtype='int32'),
    SparseFeat('day_of_week_cat',   vocabulary_size=_vocab_from_int(sparse_tr['day_of_week_cat']),   embedding_dim=EMBED_DIM, dtype='int32'),
    SparseFeat('age_group_cat',     vocabulary_size= age_vocab_size,                                 embedding_dim=EMBED_DIM, dtype='int32'),
]

sparse_hash = [
    SparseFeat('l_feat_14', vocabulary_size= HASH_BUCKET, embedding_dim= EMBED_DIM , use_hash= True ,dtype='string'),
]


dense_feats = [DenseFeat(c, 1) for c in dense_cols]


# 순서형 - sparse에 넣기
for c in map_cols:
    name = f"{c}_cat"
    if name in sparse_tr.columns:
        sparse_fixed.append(
            SparseFeat(name, vocabulary_size=_vocab_from_int(sparse_tr[name]), embedding_dim=EMBED_DIM, dtype='int32')
        )


varlen_seq  = VarLenSparseFeat(
    sparsefeat = SparseFeat('seq' ,
                            vocabulary_size = vocab_seq , # train 기준으로 vocab 계산(0=패딩/UNK 전제)
                            embedding_dim   = EMBED_DIM ,
                            # use_hash=True ,
                            dtype= 'int32', # 정수형 사용 -> 해시 안됨
                              ),
    maxlen   = MAX_LEN,
    combiner = 'mean',
    length_name = 'seq_len',
    weight_name = None,
    weight_norm = False
)




In [21]:
fixlen_feature_columns = sparse_fixed + sparse_hash + dense_feats

if varlen_seq is not None:
    linear_feature_columns = fixlen_feature_columns + [varlen_seq]
    dnn_feature_columns    = fixlen_feature_columns + [varlen_seq]
else:
    linear_feature_columns = fixlen_feature_columns
    dnn_feature_columns    = fixlen_feature_columns

feature_names = get_feature_names(linear_feature_columns + dnn_feature_columns)


# DIN: 고정길이 + 시퀀스(VarLenSparseFeat) 함께 사용

# dnn_feature_columns_din = linear_feature_columns + [varlen_seq]
# behavior_feature_list = ['inventory_id']  # query(현재 타깃)와 history 키 그룹 매칭


In [22]:
# 검증용
assert seq_tr_pad.max() < vocab_seq and seq_te_pad.max() < vocab_seq

## 학습 샘플 생성 및 모델 학습

DeepCTR 모델은 내부적으로 특성 이름별로 Input Layer를 자동 생성한다.

그래서 입력을 dict 형태로 요구한다.


<br>
주의
>

    DataFrame의 seq는 건들지 말고, 모델에 넣을 때만 seq_padded/seq_len을 넘긴다


In [23]:
def to_inputs_fast(dense_df, sparse_df, dense_cols, sparse_cols, *, seq_pad=None, seq_len=None):

    X = {}


    # Sparse: 정수 인덱스 vs 문자열 해시 구분
    for c in sparse_cols:
        s = sparse_df[c]
        if str(s.dtype).startswith("string") or s.dtype == object:
            # DeepCTR 해시용: str 배열
            X[c] = s.astype("string").to_numpy(dtype=object, copy=False)
        else:
            # 정수 인덱스
            X[c] = s.to_numpy(dtype=np.int32, copy=False)


    # Dense: float32
    for c in dense_cols:
        X[c] = dense_df[c].to_numpy(dtype=np.float32, copy=False)


    # VarLen
    if seq_pad is not None:
        X["seq"]     = np.asarray(seq_pad, dtype=np.int32)
    if seq_len is not None:
        X["seq_len"] = np.asarray(seq_len, dtype=np.int32)
    return X




X_train = to_inputs_fast(dense_tr, sparse_tr, dense_cols, sparse_cols,
                         seq_pad=seq_tr_pad if 'seq' in feature_names else None,
                         seq_len=seq_tr_len if 'seq' in feature_names else None)
X_test  = to_inputs_fast(dense_te, sparse_te, dense_cols, sparse_cols,
                         seq_pad=seq_te_pad if 'seq' in feature_names else None,
                         seq_len=seq_te_len if 'seq' in feature_names else None)

train_model_input = X_train
test_model_input  = X_test
train_y = df.loc[tr_mask, "clicked"].astype("float32").to_numpy()


In [24]:
seq_tr_pad, seq_tr_len, vocab_seq = build_seq_padded_len(
    df.loc[tr_mask, "seq"], MAX_LEN, offset=1, ignore_neg=True
)

seq_te_pad, seq_te_len, _ = build_seq_padded_len(
    df.loc[te_mask, "seq"], MAX_LEN, offset=1, ignore_neg=True
)

# train vocab 기준으로 초과 인덱스는 0(PAD/UNK) 처리
seq_tr_pad = np.where(seq_tr_pad < vocab_seq, seq_tr_pad, 0).astype("int32")
seq_te_pad = np.where(seq_te_pad < vocab_seq, seq_te_pad, 0).astype("int32")
seq_tr_len = seq_tr_len.astype("int32"); seq_te_len = seq_te_len.astype("int32")

# 1) 편의 이름 집합
hash_names  = [f.name for f in sparse_hash]     # 해시(문자열) SparseFeat들
fixed_names = [f.name for f in sparse_fixed]    # 정수 인덱스 SparseFeat들
dense_names = [f.name for f in dense_feats]

In [25]:
target = 'clicked'


train_model_input = {}
test_model_input  = {}

for name in feature_names:
    if name == "seq":
        train_model_input[name] = np.asarray(seq_tr_pad, dtype=np.int32)
        test_model_input[name]  = np.asarray(seq_te_pad, dtype=np.int32)

    elif name == "seq_len":
        train_model_input[name] = np.asarray(seq_tr_len, dtype=np.int32)
        test_model_input[name]  = np.asarray(seq_te_len, dtype=np.int32)

    elif name in hash_names:
        train_model_input[name] = sparse_tr[name].astype("string").to_numpy(dtype=object)
        test_model_input[name]  = sparse_te[name].astype("string").to_numpy(dtype=object)

    elif name in fixed_names:
        train_model_input[name] = sparse_tr[name].to_numpy(dtype=np.int32)
        test_model_input[name]  = sparse_te[name].to_numpy(dtype=np.int32)

    else:  # DenseFeat
        train_model_input[name] = dense_tr[name].to_numpy(dtype=np.float32)
        test_model_input[name]  = dense_te[name].to_numpy(dtype=np.float32)

train_y = df.loc[tr_mask, target].astype("float32").to_numpy()



In [26]:
model = DeepFM(linear_feature_columns, dnn_feature_columns, task='binary')
model.compile("adam", "binary_crossentropy",
              metrics=['binary_crossentropy'], )

In [27]:
# ============================================
# DeepCTR on Colab (Py 3.11) — TF 2.15 / Keras 2.15 호환 셋업
# ============================================
# 사용 시점:
# 1) 노트북 맨 위 셀에서 실행
# 2) DeepCTR, 모델 정의/fit 전에 반드시 실행
# --------------------------------------------
# 필요 시 설치(이미 설치되었다면 주석 유지하세요)
# !pip install -q "tensorflow==2.15.0" "keras==2.15.0" "deepctr==0.9.3"

import os, sys, types

# 레거시 tf.keras 사용 (Keras 3 경로로 빠지지 않도록)
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
import keras

# ====== 출력(버전 확인) ======
print(f"TF: {tf.__version__} | Keras: {keras.__version__}")

# --------------------------------------------
# 1) DeepCTR가 기대하는 내부 경로 심볼 매핑 (tf.keras 내부 경로 → 외부 keras 심볼 프록시)
#    - tensorflow.python.keras.layers / initializers 를 keras.* 로 연결
sys.modules["tensorflow.python.keras.layers"] = keras.layers
sys.modules["tensorflow.python.keras.initializers"] = keras.initializers

# 2) DeepCTR가 참조하는 init_ops_v2 대체 모듈 주입
from keras.initializers import TruncatedNormal, Constant, glorot_uniform
init_v2 = types.ModuleType("tensorflow.python.ops.init_ops_v2")
init_v2.TruncatedNormal = TruncatedNormal
init_v2.Constant = Constant
init_v2.glorot_uniform = glorot_uniform
sys.modules["tensorflow.python.ops.init_ops_v2"] = init_v2

# 3) TF 2.15에 없는 내부 심볼 주입:
#    model.fit 경로에서 isinstance(ds, input_lib.DistributedDatasetInterface) 체크 시 AttributeError 방지
from tensorflow.python.distribute import input_lib as _input_lib
if not hasattr(_input_lib, "DistributedDatasetInterface"):
    class DistributedDatasetInterface:
        """Minimal shim for TF 2.15; acts as a marker interface only."""
        pass
    _input_lib.DistributedDatasetInterface = DistributedDatasetInterface

print("→ Internal shims installed OK")

# --------------------------------------------
# 4) DeepCTR import 및 버전 확인
try:
    import deepctr
    from deepctr.feature_column import SparseFeat, DenseFeat, get_feature_names
    from deepctr.models import DeepFM
    print(f"DeepCTR: {deepctr.__version__} (import OK)")
except Exception as e:
    print("DeepCTR import failed:", repr(e))
    raise

# (선택) 버전 가드 — 환경이 바뀌었을 때 빨리 눈치채기 위함
def _assert_versions(tf_req="2.15.", keras_req="2.15.", deepctr_req="0.9."):
    assert tf.__version__.startswith(tf_req), f"Require TF {tf_req}x, got {tf.__version__}"
    assert keras.__version__.startswith(keras_req), f"Require Keras {keras_req}x, got {keras.__version__}"
    assert deepctr.__version__.startswith(deepctr_req), f"Require DeepCTR {deepctr_req}x, got {deepctr.__version__}"

try:
    _assert_versions()
    print("→ Version guard passed (TF/Keras/DeepCTR expected range).")
except AssertionError as ae:
    print("! Version guard warning:", ae)

print("Shim setup complete. You can now build & fit DeepCTR models safely.")


TF: 2.15.0 | Keras: 2.15.0
→ Internal shims installed OK
DeepCTR: 0.9.3 (import OK)
→ Version guard passed (TF/Keras/DeepCTR expected range).
Shim setup complete. You can now build & fit DeepCTR models safely.


In [28]:
history = model.fit(
    train_model_input,
    df.loc[tr_mask, target].to_numpy().astype("float32"),
    batch_size=256, epochs=10, verbose=2, validation_split=0.2)

pred_ans = model.predict(test_model_input, batch_size=256)



Epoch 1/10
10013/10013 - 99s - loss: 0.0914 - binary_crossentropy: 0.0914 - val_loss: 0.0889 - val_binary_crossentropy: 0.0889
Epoch 2/10
10013/10013 - 95s - loss: 0.0900 - binary_crossentropy: 0.0900 - val_loss: 0.0883 - val_binary_crossentropy: 0.0883
Epoch 3/10
10013/10013 - 98s - loss: 0.0894 - binary_crossentropy: 0.0894 - val_loss: 0.0880 - val_binary_crossentropy: 0.0880
Epoch 4/10
10013/10013 - 98s - loss: 0.0889 - binary_crossentropy: 0.0889 - val_loss: 0.0879 - val_binary_crossentropy: 0.0879
Epoch 5/10
10013/10013 - 88s - loss: 0.0885 - binary_crossentropy: 0.0885 - val_loss: 0.0878 - val_binary_crossentropy: 0.0878
Epoch 6/10
10013/10013 - 86s - loss: 0.0883 - binary_crossentropy: 0.0883 - val_loss: 0.0883 - val_binary_crossentropy: 0.0883
Epoch 7/10
10013/10013 - 85s - loss: 0.0881 - binary_crossentropy: 0.0881 - val_loss: 0.0879 - val_binary_crossentropy: 0.0879
Epoch 8/10
10013/10013 - 85s - loss: 0.0879 - binary_crossentropy: 0.0879 - val_loss: 0.0878 - val_binary_cross

롤백 or 훈련 종료 후 평가?

In [29]:
pred_ans = model.predict(test_model_input, batch_size=256)


In [30]:
import numpy as np
from sklearn.metrics import log_loss

N = len(train_y)
val_size = int(N * 0.2)
val_idx = np.arange(N - val_size, N)

val_input = {k: v[val_idx] for k, v in train_model_input.items()}
val_pred  = model.predict(val_input, batch_size=256)

print("val LogLoss(sklearn):", round(log_loss(train_y[val_idx], val_pred), 6))


val LogLoss(sklearn): 0.087764


In [33]:
p = train_y.mean()
baseline_ll = - (p*np.log(p) + (1-p)*np.log(1-p))
impr = (baseline_ll - 0.087827) / baseline_ll
print(f"Baseline LL={baseline_ll:.6f}, Relative gain={impr*100:.2f}%")


Baseline LL=0.095513, Relative gain=8.05%


In [36]:
import numpy as np
from sklearn.metrics import average_precision_score, log_loss

# --- 유틸: 학습 때 쓴 validation_split과 동일한 검증 인덱스(마지막 20%) ---
def get_val_indices(n_samples, val_frac=0.2):
    val_size = int(round(n_samples * val_frac))
    val_idx = np.arange(n_samples - val_size, n_samples)
    return val_idx

# --- 메트릭 계산 (AP, Weighted Log Loss) ---
def compute_ap_wll(y_true, y_pred, sample_weight=None):
    y_true = np.asarray(y_true).astype(np.float32).ravel()
    y_pred = np.asarray(y_pred).astype(np.float32).ravel()
    ap  = average_precision_score(y_true, y_pred, sample_weight=sample_weight)
    wll = log_loss(y_true, y_pred, sample_weight=sample_weight, labels=[0, 1])
    return ap, wll

# ====== 사용 예시 ======
# 전제: model, train_model_input, train_y 가 이미 존재
# (train_y는 df.loc[tr_mask, target].to_numpy() 등으로 만든 1D 배열)

# 1) Train 전체 점수 (선택)
train_pred = model.predict(train_model_input, batch_size=256).ravel()
# 가중치가 있으면 여기에 넣으세요: train_w = train_proc['weight'].to_numpy()
ap_tr, wll_tr = compute_ap_wll(train_y, train_pred, sample_weight=None)
print(f"[Train]  AP={ap_tr:.6f}  WLL={wll_tr:.6f}")

# 2) Validation 점수 (validation_split=0.2와 동일 구간)
val_idx = get_val_indices(len(train_y), val_frac=0.2)
val_input = {k: v[val_idx] for k, v in train_model_input.items()}
val_true  = train_y[val_idx]
val_pred  = model.predict(val_input, batch_size=256).ravel()
# 가중치가 있으면 val_w = train_w[val_idx]
ap_val, wll_val = compute_ap_wll(val_true, val_pred, sample_weight=None)
print(f"[Valid]  AP={ap_val:.6f}  WLL={wll_val:.6f}")

# 3) Test 예측 (라벨 없으므로 메트릭 계산 X, 제출용)
test_pred = model.predict(test_model_input, batch_size=256).ravel()
# 제출 예: pd.DataFrame({'ID': df.loc[te_mask,'ID'], 'clicked': test_pred}).to_csv('submission.csv', index=False)


[Train]  AP=0.085710  WLL=0.086978
[Valid]  AP=0.070463  WLL=0.087764


In [39]:
import numpy as np
from sklearn.metrics import average_precision_score, log_loss

# -----------------------------
# 1) 지표 정의
# -----------------------------
def weighted_logloss(y_true, y_prob):
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)

    # [0,1] 보장 + NaN 방지
    y_prob = np.clip(y_prob, 1e-7, 1 - 1e-7)

    pos = (y_true == 1).sum()
    neg = (y_true == 0).sum()
    w1 = 0.5 / max(pos, 1)   # 양성 총합 가중이 0.5가 되도록
    w0 = 0.5 / max(neg, 1)   # 음성 총합 가중이 0.5가 되도록
    w  = np.where(y_true == 1, w1, w0)

    return log_loss(y_true, y_prob, sample_weight=w, labels=[0, 1])

def dacon_score(y_true, y_prob):
    ap  = average_precision_score(y_true, y_prob)
    wll = weighted_logloss(y_true, y_prob)
    score = 0.5 * ap + 0.5 * (1.0 - wll)  # 리더보드 결합 스코어 형태
    return score, ap, wll


# =========================================================
# (A) y_valid, p_valid가 이미 있을 때
# =========================================================
# 예시:
# y_valid = valid_df["clicked"].to_numpy()
# p_valid = model.predict(valid_input, batch_size=256).ravel()

# s, ap, wll = dacon_score(y_valid, p_valid)
# print(f"[Valid] score={s:.6f}  AP={ap:.6f}  WLL={wll:.6f}")


# =========================================================
# (B) validation_split=0.2 로 학습했을 때 (마지막 20%를 검증으로 사용)
#    - train_model_input: 학습에 사용한 입력 딕셔너리(각 value는 같은 길이의 배열)
#    - train_y: 학습 타깃 (array)
#    - model: 학습 완료된 모델
# =========================================================
def eval_with_tail_split(train_model_input, train_y, model, val_ratio=0.2, batch_size=256):
    # 길이 N 추출 (딕셔너리 아무 키나)
    any_key = next(iter(train_model_input.keys()))
    N = len(train_model_input[any_key])

    val_size = int(round(N * val_ratio))
    val_idx = slice(N - val_size, N)

    # 검증 입력/정답 구성(마지막 20%)
    val_input = {k: v[val_idx] for k, v in train_model_input.items()}
    y_valid   = np.asarray(train_y[val_idx], dtype=int)

    # 예측 확률
    p_valid = model.predict(val_input, batch_size=batch_size).ravel().astype("float32")

    # 점수 계산
    s, ap, wll = dacon_score(y_valid, p_valid)
    print(f"[Valid] score={s:.6f}  AP={ap:.6f}  WLL={wll:.6f}")
    return s, ap, wll, y_valid, p_valid

# 사용 예:
s, ap, wll, yv, pv = eval_with_tail_split(train_model_input, train_y, model, val_ratio=0.2, batch_size=256)


[Valid] score=-0.384476  AP=0.070463  WLL=1.839414


In [42]:
import numpy as np
from sklearn.metrics import average_precision_score, log_loss

# -----------------------------
# 1) 대회 WLL, 로컬 리포트 함수
# -----------------------------
def weighted_logloss(y_true, y_prob):
    """WLL: 양/음성 기여를 0.5/0.5로 균형화한 LogLoss."""
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_prob = np.clip(y_prob, 1e-7, 1-1e-7)

    pos = int((y_true == 1).sum())
    neg = int((y_true == 0).sum())
    w1  = 0.5 / max(pos, 1)
    w0  = 0.5 / max(neg, 1)
    w   = np.where(y_true == 1, w1, w0)

    return log_loss(y_true, y_prob, sample_weight=w, labels=[0, 1])

def dacon_local_report(y_true, y_prob, title="[Valid]"):
    """AP, WLL, (참고) 비가중 LL, 로컬 결합 점수까지 출력."""
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float).ravel()
    y_prob = np.clip(y_prob, 1e-7, 1-1e-7)

    ap  = average_precision_score(y_true, y_prob)
    wll = weighted_logloss(y_true, y_prob)
    ll  = log_loss(y_true, y_prob, labels=[0,1])

    # 로컬 참고용 결합 스코어(공식 내부 변환식과 동일하다고 보장 X)
    local_score = 0.5 * ap + 0.5 * (1.0 - wll)

    n = len(y_true); pos = int((y_true==1).sum()); neg = n - pos
    print(f"{title} n={n}  pos={pos}  neg={neg}")
    print(f"  AP  = {ap:.6f}")
    print(f"  WLL = {wll:.6f}  (weighted logloss; class balance 50:50)")
    print(f"  LL  = {ll:.6f}   (unweighted logloss; 참고)")
    print(f"  Local combined ≈ 0.5*AP + 0.5*(1-WLL) = {local_score:.6f}")
    print(f"  prob range = [{y_prob.min():.6f}, {y_prob.max():.6f}]  "
          f"%0/1-hard={(np.mean((y_prob<=1e-6)|(y_prob>=1-1e-6))*100):.2f}%")

# -----------------------------
# 2) validation_split로 학습했을 때 평가 헬퍼
# -----------------------------
def eval_with_tail_split(train_model_input, train_y, model, val_ratio=0.2, batch_size=256):
    """
    Keras의 validation_split=0.2 를 썼다면 '입력의 마지막 20%'가 검증셋입니다.
    (Keras는 분할 후에만 학습 쪽 셔플을 적용)
    """
    any_key = next(iter(train_model_input.keys()))
    N = len(train_model_input[any_key])
    val_size = int(round(N * val_ratio))
    val_idx = slice(N - val_size, N)

    val_input = {k: v[val_idx] for k, v in train_model_input.items()}
    y_valid   = np.asarray(train_y[val_idx], dtype=int)

    p_valid = model.predict(val_input, batch_size=batch_size).ravel().astype("float32")
    dacon_local_report(y_valid, p_valid, "[Valid]")
    return y_valid, p_valid

# =============================
# (A) 이미 y_valid, p_valid가 있을 때
# =============================
# 예)
# dacon_local_report(y_valid, p_valid, "[Valid]")

# =============================
# (B) validation_split 사용 시
# =============================
# 예)
yv, pv = eval_with_tail_split(train_model_input, train_y, model, val_ratio=0.2, batch_size=256)


[Valid] n=640777  pos=12171  neg=628606
  AP  = 0.070463
  WLL = 1.839414  (weighted logloss; class balance 50:50)
  LL  = 0.087764   (unweighted logloss; 참고)
  Local combined ≈ 0.5*AP + 0.5*(1-WLL) = -0.384476
  prob range = [0.000000, 0.999733]  %0/1-hard=0.00%


모델 저장

In [ ]:
save_dir = "/content/drive/MyDrive/Colab Notebooks/CRT/model"
import os
os.makedirs(save_dir, exist_ok=True)

# 저장: 체크포인트 형식(.index, .data-00000-of-00001 파일 세트)
ckpt_path = f"{save_dir}/best.weights"
model.save_weights(ckpt_path)          # 또는 model.save_weights(ckpt_path, save_format='tf')


In [ ]:
# # 로드: 동일한 모델 구조를 코드로 재생성한 뒤
model2 = DeepFM(linear_feature_columns, dnn_feature_columns, task='binary')
model2.compile(optimizer='adam', loss='binary_crossentropy')
model2.load_weights(ckpt_path)

제출 파일 만들기

In [34]:
import re

test_ids = df.loc[te_mask, "ID"].astype(str)

# (a) 포맷 검사
bad_fmt = ~test_ids.str.match(r"^TEST_\d{7}$")
print("잘못된 ID 포맷 개수:", int(bad_fmt.sum()))
print(df.loc[te_mask].loc[bad_fmt, "ID"].head())

# (b) 중복 검사
dup = test_ids.duplicated(keep=False)
print("ID 중복 개수:", int(dup.sum()))

# (c) 연속성(누락) 추정: 숫자부분 분석
id_num = test_ids.str.extract(r"(\d{7})$", expand=False).astype(int)
min_id, max_id = int(id_num.min()), int(id_num.max())
expected_cnt = (max_id - min_id + 1)
print("ID 최소/최대:", min_id, max_id, "연속이라면 기대 행수:", expected_cnt)
print("지금 테스트 행수:", int(len(test_ids)))

# 어떤 숫자들이 빠졌는지 대략 확인
missing_nums = sorted(set(range(min_id, max_id + 1)) - set(id_num.to_list()))
print("누락된 ID 숫자 수:", len(missing_nums))
if missing_nums[:5]:
    print("예시 누락 숫자:", missing_nums[:5])


잘못된 ID 포맷 개수: 0
Series([], Name: ID, dtype: object)
ID 중복 개수: 0
ID 최소/최대: 0 1527297 연속이라면 기대 행수: 1527298
지금 테스트 행수: 1527298
누락된 ID 숫자 수: 0


In [35]:
# 최종 예측 직전에 실제로 사용하는 test 인덱스/ID를 저장해서 비교
test_idx_input = None
for k,v in test_model_input.items():
    test_idx_input = len(v)  # 길이가 모두 같아야 함
    break
print("test_model_input length:", test_idx_input)

# df에서 te_mask True인 행 수와 맞는지
print("df[te_mask] length:", int(te_mask.sum()))


test_model_input length: 1527298
df[te_mask] length: 1527298


In [37]:
# (앞에서 만든 is_train_num 기반 te_mask 사용)
test_idx = df.index[te_mask]  # 원래 순서 유지가 안전

ids  = df.loc[test_idx, "ID"].astype(str).to_numpy()
prob = np.asarray(pred_ans).reshape(-1)

# 길이 일치 확인
assert len(ids) == len(prob), f"길이 불일치: ids={len(ids)}, pred={len(prob)}"

# 확률 안전화
prob = np.clip(prob, 1e-7, 1-1e-7).astype("float32")
if np.isnan(prob).any() or np.isinf(prob).any():
    raise ValueError("pred_ans에 NaN/Inf가 있습니다.")

# 제출 파일 생성 (ID, clicked 두 컬럼만)
sub = pd.DataFrame({"ID": ids, "clicked": prob})
sub.to_csv("submission.csv", index=False)  # 순서 유지, 헤더 포함

print(sub.shape, sub.dtypes)
print(sub.head())


(1527298, 2) ID          object
clicked    float32
dtype: object
             ID   clicked
0  TEST_0000000  0.008706
1  TEST_0000001  0.011074
2  TEST_0000002  0.016718
3  TEST_0000003  0.021054
4  TEST_0000004  0.003132
